# ConverterSmartGrid: entrenar en Colab Pro

Opción del entrenamiento local (spec 113, RF-16): ejecuta aquí **las mismas órdenes de `tools/`** que en el equipo,
con el paquete que preparó el equipo, y deja el candidato en tu Drive. El candidato vuelve al equipo y pasa allí por
las mismas puertas que lo entrenado en local. Nada de aquí es necesario para entrenar en el equipo.

1. En el equipo: `python tools/entrenamiento_remoto.py preparar --familia asistente_redaccion --dataset <carpeta>`
   (o el botón «Entrenar en Colab» de la web). Espera a que Drive para escritorio sincronice el paquete.
2. Aquí: en «Entorno de ejecución → Cambiar tipo» elige **A100** (o H100 o L4) y **RAM alta**; escribe `PAQUETE` y
   ejecuta todas las celdas en orden. Con A100 se entrenan tres épocas y tres semillas, cada semilla un candidato; con
   una T4 se entrenaría lo mismo que en el equipo y Colab no aportaría. Desde VS Code, abre este mismo
   fichero con la extensión de Colab y elige un núcleo de Colab: el resultado es el mismo (RF-14).
3. Si Colab corta la sesión, vuelve a ejecutar todo: se retoma desde el último punto de control en Drive (RF-7).
4. En el equipo: `python tools/entrenamiento_remoto.py traer` y la orden de la puerta que escriba.

Este cuaderno no lleva datos, salidas ni credenciales (RF-1). Si el repositorio fuese privado, el token va en los
secretos de Colab con el nombre `GITHUB_TOKEN`, nunca aquí (RF-2).

In [ ]:
#@title 1. Parámetros
PAQUETE = ''  #@param {type:"string"}
EPOCAS = 0  #@param {type:"integer"}
SEMILLAS = ''  #@param {type:"string"}
# 0 y vacío: los decide la GPU de Colab (er.perfil_colab). Para fijarlos: EPOCAS = 2, SEMILLAS = '11,12'.
EXPERIMENTO = '072-T9b'  #@param {type:"string"}

In [ ]:
#@title 2. Drive, paquete y el código en la revisión exacta del paquete (RF-2)
import json, os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/ConverterSmartGrid/colab')
disponibles = sorted(p.name for p in (BASE / 'paquetes').iterdir() if (p / 'manifiesto.json').is_file())
if PAQUETE not in disponibles:
    raise SystemExit('Escribe PAQUETE (celda 1) con uno de estos: ' + ', '.join(disponibles))
CARPETA = BASE / 'paquetes' / PAQUETE
MANIFIESTO = json.loads((CARPETA / 'manifiesto.json').read_text(encoding='utf-8'))
REVISION, FAMILIA = MANIFIESTO['revision'], MANIFIESTO['familia']

REPO = Path('/content/ConverterSmartGrid')
url = f"https://github.com/{MANIFIESTO['repositorio']}.git"   # el repositorio del que salió el paquete
try:
    from google.colab import userdata
    url = url.replace('https://', f"https://x-access-token:{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass                                          # repositorio público: sin token
if not REPO.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', url, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--quiet', '--detach', REVISION], check=True)
cabeza = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert cabeza == REVISION, f'el código es {cabeza[:8]} y el paquete pide {REVISION[:8]}'
print(f'Código en {REVISION[:8]}; familia {FAMILIA}; {len(MANIFIESTO["ficheros"])} ficheros en el paquete.')

In [ ]:
#@title 3. Versiones fijadas del proyecto y paquete verificado (RF-3)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[ml]'], check=True)
from importlib import metadata
distintas = []
for linea in (REPO / 'requirements.txt').read_text(encoding='utf-8').splitlines():
    linea = linea.split('#')[0].split(';')[0].strip()
    if '==' in linea:
        nombre, version = (x.strip() for x in linea.split('==', 1))
        try:
            instalada = metadata.version(nombre)
        except metadata.PackageNotFoundError:
            instalada = None
        if instalada != version:
            distintas.append(f'{nombre}: pide {version}, hay {instalada}')
print('Todas las versiones fijadas están.' if not distintas else 'Distintas o ausentes:\n' + '\n'.join(distintas))
sys.path.insert(0, str(REPO / 'src'))
from convertersmartgrid.aplicacion import entrenamiento_remoto as er
er.verificar_paquete(CARPETA)                     # cada fichero con su huella antes de entrenar
print('Paquete verificado.')

In [ ]:
#@title 4. Perfil de Colab Pro y entrenamiento con la misma orden que en el equipo (RF-7, RF-8, RF-16)
import platform, shlex, torch
nombre_gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
vram = torch.cuda.get_device_properties(0).total_memory / 2**30 if nombre_gpu else None
PERFIL = er.perfil_colab(nombre_gpu, vram)
epocas = EPOCAS or PERFIL['epocas']
semillas = [int(s) for s in SEMILLAS.split(',') if s.strip()] or PERFIL['semillas']
print(f"{PERFIL['gpu']} ({PERFIL['vram_gib']} GiB): {PERFIL['nota']}. Épocas {epocas}, semillas {semillas}.")
entorno = {'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
           'gpu': nombre_gpu, 'vram_gib': PERFIL['vram_gib']}
for semilla in semillas:
    SALIDA = BASE / 'candidatos' / f'{PAQUETE}_e{epocas}_s{semilla}'
    if (SALIDA / 'candidato.json').is_file():
        print(f'{SALIDA.name}: ya terminado.')
        continue
    SALIDA.mkdir(parents=True, exist_ok=True)
    orden = er.orden_de_entrenamiento(FAMILIA, datos=str(CARPETA / 'datos'), salida=str(SALIDA), semilla=semilla,
                                      epocas=epocas)
    args = shlex.split(orden) + ['--experimento-autorizado', EXPERIMENTO]
    puntos = sorted(SALIDA.rglob('progreso_*'))
    if puntos and 'reanudar' in er.FAMILIAS[FAMILIA]:
        args += shlex.split(er.FAMILIAS[FAMILIA]['reanudar'].format(punto=puntos[-1]))
        print(f'{SALIDA.name}: se retoma desde {puntos[-1].name}.')
    print('Orden:', ' '.join(args))
    subprocess.run([sys.executable, *args], cwd=REPO, check=True)
    # RF-9: el candidato con su revisión, paquete, semilla, entorno y métricas; sin candidato.json no se trae.
    metricas = {}
    for nombre in ('metricas.json', 'metrics.json', 'resumen.json'):
        for ruta in SALIDA.rglob(nombre):
            metricas[ruta.relative_to(SALIDA).as_posix()] = json.loads(ruta.read_text(encoding='utf-8'))
    m = er.manifiesto_candidato(familia=FAMILIA, revision=REVISION, paquete=MANIFIESTO['huella'], semilla=semilla,
                                entorno=entorno, hiperparametros={'epocas': epocas, 'experimento': EXPERIMENTO},
                                metricas=metricas, carpeta=SALIDA,
                                pruebas=[{'semilla': s, 'epocas': epocas} for s in semillas])
    (SALIDA / 'candidato.json').write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Candidato listo en Drive: {SALIDA.name} ({len(m["artefactos"])} artefactos).')
print('En el equipo: python tools/entrenamiento_remoto.py traer')